# Rice, radius, and protest figures

Generates exactly four outputs: `rices_grids_150dpi_q75.pdf`, `acs_grids_radius12km.png`, `protests_monthly_bars.png`, and `ac_protest_plot.png`.

In [ ]:
from pathlib import Path
import geopandas as gpd
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import ListedColormap

PROJECT = Path(r"C:\Users\eunic\Dropbox\sa_fires\proj_bureaucrats_farms")
SHARED = PROJECT.parent
INTERMEDIATE = PROJECT / "data_output/intermediate"
FIGURES = PROJECT / "tex/paper/figures"
FIGURES.mkdir(parents=True, exist_ok=True)
MASTER = INTERMEDIATE / "0_master_dataset.parquet"
ACS_PATH = INTERMEDIATE / "_0_2_3_ACs_right_shapefile.shp"
GRID_PATH = INTERMEDIATE / "1-grid-generation.shp"
RICE_PATH = SHARED / "data/input/crop-production-2010-mapspam/spam2010v2r0_global_prod.csv/spam2010V2r0_global_P_TA.csv"
PROTEST_PATH = SHARED / "data/input/acled/2000-01-01-2025-05-13-South_Asia-India.csv"

for path in [MASTER, ACS_PATH, GRID_PATH, RICE_PATH, PROTEST_PATH]:
    if not path.exists():
        raise FileNotFoundError(path)

acs = gpd.read_file(ACS_PATH)
acs["geometry"] = acs.geometry.make_valid()
grid = gpd.read_file(GRID_PATH).rename(columns={"unq_s__": "unique_small_grid_id"})
grid = grid[["unique_small_grid_id", "geometry"]].drop_duplicates("unique_small_grid_id")
if grid.crs != acs.crs:
    grid = grid.to_crs(acs.crs)

## Rice-production grid map

In [ ]:
sample_ids = pd.read_parquet(
    MASTER, columns=["unique_small_grid_id"],
    filters=[("year", "==", 2020), ("month", "==", 12)],
).drop_duplicates()
sample_grid = gpd.GeoDataFrame(sample_ids.merge(grid, on="unique_small_grid_id", how="inner"), geometry="geometry", crs=grid.crs)

rice = pd.read_csv(RICE_PATH, encoding="windows-1252", usecols=["x", "y", "rice_a"])
rice = rice.dropna(subset=["x", "y", "rice_a"])
rice_points = gpd.GeoDataFrame(rice, geometry=gpd.points_from_xy(rice["x"], rice["y"]), crs="EPSG:4326")
rice_points = gpd.sjoin(rice_points, acs[["ac_uq_id", "geometry"]], how="inner", predicate="within")
rice_points["quintile"] = pd.qcut(rice_points["rice_a"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
colors = ["#fee8c8", "#fdbb84", "#fc8d59", "#e34a33", "#b30000"]
cmap = ListedColormap(colors)

fig, ax = plt.subplots(figsize=(15, 15))
sample_grid.boundary.plot(ax=ax, color="#b5bac3", linewidth=0.12, zorder=1)
rice_points.plot(ax=ax, column="quintile", categorical=True, cmap=cmap, markersize=2.0, linewidth=0, rasterized=True, zorder=10)
ax.legend(handles=[mpatches.Patch(color=colors[i - 1], label=f"Q{i}") for i in range(1, 6)], title="Quintiles", loc="upper right", frameon=False)
ax.set_axis_off()
rice_output = FIGURES / "rices_grids_150dpi_q75.pdf"
fig.savefig(rice_output, format="pdf", dpi=150, bbox_inches="tight", metadata={"Title": "Rice production quintiles"})
plt.close(fig)
print(f"Saved {rice_output}")

## AC grids with a 12 km radius

In [ ]:
radius_ac = 61
radius_grid = 7334
radius_ids = pd.read_parquet(
    MASTER, columns=["unique_small_grid_id"], filters=[("ac_uq_id", "==", radius_ac)]
).drop_duplicates()
selected = gpd.GeoDataFrame(radius_ids.merge(grid, on="unique_small_grid_id", how="inner"), geometry="geometry", crs=grid.crs)
focus = selected.loc[selected["unique_small_grid_id"].eq(radius_grid)]
if len(focus) != 1:
    raise ValueError(f"Expected one grid {radius_grid}; found {len(focus)}")
focus_metric = focus.to_crs(7755)
center_metric = focus_metric.geometry.iloc[0].centroid
center = gpd.GeoSeries([center_metric], crs=7755).to_crs(selected.crs)
buffer = gpd.GeoSeries([center_metric.buffer(12_000)], crs=7755).to_crs(selected.crs)

fig, ax = plt.subplots(figsize=(10, 8))
selected.plot(ax=ax, edgecolor="black", facecolor="none", linewidth=0.5)
acs.loc[acs["ac_uq_id"].eq(radius_ac)].boundary.plot(ax=ax, edgecolor="black", linewidth=1.5)
buffer.boundary.plot(ax=ax, edgecolor="black", linewidth=1)
center.plot(ax=ax, color="black", marker="s", markersize=40)
ax.set_axis_off()
fig.tight_layout()
radius_output = FIGURES / "acs_grids_radius12km.png"
fig.savefig(radius_output, dpi=300, bbox_inches="tight")
plt.close(fig)
print(f"Saved {radius_output}")

## Farmer-protest data and monthly bars

In [ ]:
protests = pd.read_csv(PROTEST_PATH, sep=";", low_memory=False)
protests["event_date"] = pd.to_datetime(protests["event_date"], dayfirst=True)
protests = protests.loc[protests["event_date"].between("2020-06-01", "2021-12-09")].copy()
farmer = protests["assoc_actor_1"].str.lower().str.contains(r"\bfarm\w*", na=False)
farmer |= protests["notes"].str.lower().str.contains(r"\bfarm\w*", na=False)
farmer_protests = protests.loc[farmer].copy()
farmer_protests["year_month"] = farmer_protests["event_date"].dt.to_period("M")
monthly = farmer_protests[["year_month", "admin1"]].drop_duplicates()["year_month"].value_counts().sort_index()
monthly.index = monthly.index.to_timestamp()

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(monthly.index, monthly.values, color="steelblue", edgecolor="black", width=20)
ax.set_xlabel("Month")
ax.set_ylabel("Count of Number of Protests")
ax.grid(alpha=0.3, axis="y")
span = monthly.index.max() - monthly.index.min()
ax.set_xlim(monthly.index.min() - span * 0.05, monthly.index.max() + span * 0.05)
fig.tight_layout()
bars_output = FIGURES / "protests_monthly_bars.png"
fig.savefig(bars_output, dpi=300)
plt.close(fig)
print(f"Saved {bars_output}")

## Protest buffers in the selected AC

In [ ]:
protest_points = gpd.GeoDataFrame(
    farmer_protests, geometry=gpd.points_from_xy(farmer_protests["longitude"], farmer_protests["latitude"]), crs="EPSG:4326"
)
events = gpd.sjoin(protest_points, acs[["ac_uq_id", "geometry"]], how="inner", predicate="within")
protest_ac = 285
events = events.loc[events["ac_uq_id"].eq(protest_ac)].drop_duplicates(["latitude", "longitude"]).copy()
buffers = events.to_crs(7755).copy()
buffers.geometry = buffers.geometry.buffer(5_000)
buffers = buffers.to_crs(acs.crs)

fig, ax = plt.subplots(figsize=(8, 8))
buffers.plot(ax=ax, color="blue", linewidth=0.7, alpha=0.2)
ax.scatter(events["longitude"], events["latitude"], color="black", marker="o", s=40, zorder=5)
acs.loc[acs["ac_uq_id"].eq(protest_ac)].boundary.plot(ax=ax, color="black", linewidth=1.0)
ax.legend(handles=[
    plt.Line2D([], [], color="black", marker="o", linestyle="None", markersize=8, label="Protest Location"),
    mpatches.Patch(color="blue", alpha=0.2, label="Protest Buffer Area"),
    mpatches.Patch(edgecolor="black", facecolor="none", label="Assembly Constituency Border"),
], loc="lower right", fontsize=12, frameon=False)
ax.set_axis_off()
fig.tight_layout()
ac_output = FIGURES / "ac_protest_plot.png"
fig.savefig(ac_output, dpi=300, bbox_inches="tight")
plt.close(fig)
print(f"Saved {ac_output}")